# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
md = dataset.metadata  # Metadata is an object, not a dict
print(f"Dataset name: {getattr(md, 'name', None)}")
print(f"Description: {getattr(md, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's explore the available record sets and their schema. **All IDs used in further steps reference the `@id` property in the schema.**

In [ ]:
# List RecordSets, Fields, and Columns by their @id

rs_list = list(dataset.record_sets())
print("Record Sets:")
for rs in rs_list:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  > Name: {rs.get('name')}")
    # Show fields
    for field in rs.get('fields', []):
        print(f"    - Field @id: {field['@id']} (name={field.get('name')}, dataType={field.get('dataType')})")
        # If field has columns
        if 'columns' in field:
            for col in field['columns']:
                print(f"      * Column @id: {col['@id']} (name={col.get('name')})")
    print("")

## 3. Data Extraction
Load data from the principal record set(s) into DataFrames for further analysis. Use the record set and field `@id`s from the overview above.

**Note:** Replace the examples below with the relevant record set(s) and field `@id`s for your analysis.

In [ ]:
# Identify the main record set and field IDs
# Example RecordSet/fields: change these if your dataset overview lists others

# Here we dynamically detect the main record set(s) for typical tabular survey data
rs_list = list(dataset.record_sets())
# Use any RecordSet with fields and columns, usually a data table
main_record_sets = []
for rs in rs_list:
    if rs.get('fields'):
        main_record_sets.append(rs['@id'])

print(f"Discovered main record sets: {main_record_sets}")

# Extract data for each record set
dataframes = {}
for rs_id in main_record_sets:
    print(f"Loading records from RecordSet @id={rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in RecordSet {rs_id}:", df.columns.tolist())

# For convenience, pick the first main record set for further processing
main_rs_id = main_record_sets[0]
print(f"\nPreview of first 5 rows from RecordSet {main_rs_id}:")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: e.g., filtering records, normalization, grouping. All column access is by their field or column `@id`.

In [ ]:
# Identify likely numeric fields for EDA
# We'll print the numeric fields using @id (from the schema)

fields = None
for rs in rs_list:
    if rs['@id'] == main_rs_id:
        fields = rs.get('fields', [])

numeric_fields = []
for field in fields:
    # 'dataType' could be 'Float', 'Integer', etc.
    dt = field.get('dataType', '').lower()
    if 'float' in dt or 'int' in dt or 'number' in dt:
        numeric_fields.append(field['@id'])

if len(numeric_fields) == 0:
    print("No numeric fields detected. Using all columns that look numeric in the DataFrame.")
    # Fallback: look for numeric dtypes in the DataFrame itself
    numeric_fields = dataframes[main_rs_id].select_dtypes(include=['number']).columns.tolist()
else:
    print(f"Numeric fields detected by @id: {numeric_fields}")

# Choose a numeric field for demonstration
numeric_field_id = numeric_fields[0] if numeric_fields else None

if numeric_field_id:
    df = dataframes[main_rs_id]
    print(f"Summary statistics for field '{numeric_field_id}':\n", df[numeric_field_id].describe())
    
    # Example: Filter records with value above threshold
    threshold = df[numeric_field_id].mean()  # Use mean as a threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
    
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Try grouping by a relevant categorical field (e.g. gender, education, etc.)
    # Find a suitable group field (string/categorical)
    candidate_group_fields = [f['@id'] for f in fields if f.get('dataType', '').lower() == 'text']
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions of the main numeric field, or compare across groups.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8, 5))
    plt.hist(df[numeric_field_id].dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we have found a group field
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and analyze a FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library. 

- We loaded and explored the dataset structure using its Croissant schema.
- We identified record sets, their fields, and accessed data using the canonical `@id` identifiers as recommended by Croissant.
- Simple exploratory analysis and visualization on numeric fields was performed.

You can extend this notebook by analyzing additional fields, building machine learning models, or generating more advanced visualizations as needed.